# HBA-RL: Matched Flat PPO under load (ABL-4 addition) on Kaggle

Trains the capacity-matched Flat PPO (ABL-1 configuration) at arrival probabilities 0.6 and 0.9, on both datasets and all 10 seeds (40 runs). It then re-runs the evaluation, statistics, tables and figures over all runs, so the final bundle is complete.

**Setup**
1. Inputs: the code dataset (`hba-rl-code`), the Criteo dataset (`criteo-200k`), and the **output of the Run 2 notebook** (`hba-rl-run2`), which holds all earlier runs. Do not add the first notebook's output.
2. *Internet* **On**, accelerator **None**.
3. **Save Version → Save & Run All (Commit)**. Expected time: about 2–3 hours.

Results are in the *Output* tab (`results_bundle.zip`).

## 1. Parameters

In [ ]:
import os
os.environ["SEEDS"] = "7 42 123 456 11 22 33 44 55 66"
os.environ["DATASETS"] = "synth criteo"
os.environ["EPISODES"] = "2000"
os.environ["WORKERS"] = str(os.cpu_count())
print(os.environ["DATASETS"], os.environ["WORKERS"], "workers")

## 2. Code and data

In [ ]:
import glob, os, shutil, subprocess, hashlib
src = os.path.dirname(glob.glob("/kaggle/input/**/reproduce.sh", recursive=True)[0])
shutil.rmtree("/kaggle/working/HBA-RL", ignore_errors=True)
shutil.copytree(src, "/kaggle/working/HBA-RL")
os.chdir("/kaggle/working/HBA-RL")
raw = max(glob.glob("/kaggle/input/**/train.txt", recursive=True), key=os.path.getsize)
print("Criteo source:", raw, round(os.path.getsize(raw) / 1e6), "MB")
subprocess.run(["python", "data/make_criteo_extract.py", "--src", raw, "--out", "criteo"], check=True)
expected = "69824b15277d6dd1cfc1bba0ab9e2874f11556942e86fbaaa04a0514c18d9bd0"
h = hashlib.sha256(open("criteo/train.txt", "rb").read()).hexdigest()
print("MATCHES the manuscript extract" if h == expected else "DIFFERENT from the manuscript extract: " + h)

In [ ]:
!python data/make_criteo_extract.py --check criteo

## 3. Environment
Pinned versions. Expected output of the check: `1.26.4 2.5.1 0.29.1 1.14.1 2.2.3`.

In [ ]:
%%writefile requirements.txt
numpy==1.26.4
pandas==2.2.3
scipy==1.14.1
matplotlib==3.9.2
torch==2.5.1
gymnasium==0.29.1
tqdm==4.67.1
pyyaml==6.0.2


In [ ]:
!pip install -q -r requirements.txt
!python -c "import numpy, torch, gymnasium, scipy, pandas; print(numpy.__version__, torch.__version__.split('+')[0], gymnasium.__version__, scipy.__version__, pandas.__version__)"

## 4. Resume from a previous version (optional)
If an earlier version of this notebook is added as an input, its completed runs are restored and skipped.

In [ ]:
import glob, shutil, os
for d in glob.glob("/kaggle/input/**/businessAware", recursive=True):
    shutil.copytree(d, "businessAware", dirs_exist_ok=True)
    print("restored", d)
    res = os.path.join(os.path.dirname(d), "results")
    if os.path.isdir(res):
        shutil.copytree(res, "results", dirs_exist_ok=True)
        print("restored", res)
os.makedirs("businessAware", exist_ok=True); os.makedirs("results", exist_ok=True)
print(len(glob.glob("businessAware/**/training_summary.json", recursive=True)), "finished runs restored")

## 5. Training (parallel over seeds)

In [ ]:
%%writefile run_parallel.py
import os, sys, glob, subprocess, time, shutil, concurrent.futures as cf
ROOT = os.getcwd()
SEEDS = [int(s) for s in os.environ["SEEDS"].split()]
DATASETS = os.environ["DATASETS"].split()
EPS = int(os.environ["EPISODES"])
WORKERS = int(os.environ["WORKERS"])
env = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1")
os.makedirs("results/logs/jobs", exist_ok=True)

def chain(ds, s):
    c = ["criteo"] if ds == "criteo" else []
    flat = ["python", "scripts/train_flat_ppo.py"] + c
    base = ["--seed", str(s), "--episodes", str(EPS), "--matched"]
    return [(f"ABL-4 matched flat p={p}", flat + base + ["--arrival-prob", p, "--tag", f"abl4_p{p}"]) for p in ("0.6", "0.9")]

def job(ds, s):
    out = []
    with open(f"results/logs/jobs/{ds}_seed{s}.txt", "a") as log:
        for name, cmd in chain(ds, s):
            t0 = time.time()
            rc = subprocess.run(cmd, cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT).returncode
            out.append(f"  [{int(time.time()-t0)} s] {name} {ds} seed {s}" + ("" if rc == 0 else f"  FAILED rc={rc}"))
            if rc != 0: break
    return ds, s, out

jobs = [(d, s) for d in DATASETS for s in SEEDS]
t0 = time.time(); failed = []
with cf.ThreadPoolExecutor(WORKERS) as ex:
    for f in cf.as_completed([ex.submit(job, d, s) for d, s in jobs]):
        ds, s, out = f.result()
        print(f"--- finished {ds} seed {s}  (elapsed {int((time.time()-t0)/60)} min)")
        print("\n".join(out))
        if any("FAILED" in o for o in out): failed.append((ds, s))
print("FAILED:", failed if failed else "none")


In [ ]:
!python run_parallel.py 2>&1 | tee -a results/reproduction_log.txt

## 6. Evaluation, statistics, tables and figures

In [ ]:
if "criteo" in os.environ["DATASETS"]:
    get_ipython().system("python scripts/eval_all.py criteo 2>&1 | tee -a results/reproduction_log.txt")
else:
    print("Evaluation skipped: it runs in the Criteo run (Run 2).")

## 7. Verification
Checks that every evaluation log has 100 episodes with seeds 42–141 and prints the main results table.

In [ ]:
%%writefile verify.py
import glob, pandas as pd
pd.set_option("display.width", 220)
logs = glob.glob("results/logs/eval_*.csv")
bad = [f for f in logs if len(pd.read_csv(f)) != 100 or list(pd.read_csv(f).episode_seed) != list(range(42, 142))]
print(len(logs), "evaluation logs; problems:", bad if bad else "none")
print(pd.read_csv("results/tables/R1_main_results.csv").round(2).to_string(index=False))

In [ ]:
if "criteo" in os.environ["DATASETS"]:
    get_ipython().system("python verify.py")
else:
    print("Verification runs in Run 2.")

## 8. Export

In [ ]:
!cd /kaggle/working/HBA-RL && zip -qr /kaggle/working/results_bundle.zip results businessAware -x '*.pt'